# Car insurance PDF preprocessing notebook

## Purpose and scope
This notebook preprocesses **Dutch car insurance policy PDFs** from `car_pdfs/` into document/page/chunk outputs used by downstream retrieval and summary steps.

Scope is intentionally limited to preprocessing:
- no zip extraction workflow
- no mixed insurance dataset logic
- no retrieval / RAG / ColBERT code

## Environment and imports
Install notebook dependencies in Colab if needed, then import runtime libraries.

In [ ]:
# Uncomment in Colab when dependencies are missing.
# !pip -q install pymupdf pandas pyarrow tqdm matplotlib spacy
# !python -m spacy download nl_core_news_sm

import json
import math
import re
import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import fitz  # PyMuPDF
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

try:
    import spacy
    SPACY_AVAILABLE = True
except ImportError:
    SPACY_AVAILABLE = False

## Configuration
Set paths and preprocessing thresholds. The defaults target car-only processing with overlapping chunks.

In [ ]:
CONFIG: Dict[str, Any] = {
    "PDF_DIR": "./car_pdfs",
    "OUT_DIR": "./outputs_car_preprocessing",
    "SAVE_FULL_PAGE_TEXT": False,
    "DOC_MIN_TOTAL_CHARS": 2000,
    "PAGE_LOW_TEXT_CHARS": 200,
    "MAX_LOW_TEXT_PAGE_FRAC": 0.5,
    "REPEATED_LINE_PAGE_FRAC": 0.5,
    "REPEATED_LINE_MAX_CHARS": 80,
    "BLOCK_MERGE_VERTICAL_GAP": 12.0,
    "BLOCK_MERGE_X_TOL": 20.0,
    "CHUNK_TARGET_CHARS": 1200,
    "CHUNK_MIN_CHARS": 400,
    "CHUNK_OVERLAP_CHARS": 250,
    "ALLOW_CROSS_PAGE_CHUNKS": False,
    "MAX_CHARS_FOR_SPACY": 2000,
    "RANDOM_SEED": 42,
}

np.random.seed(CONFIG["RANDOM_SEED"])

OUT_DIR = Path(CONFIG["OUT_DIR"])
(OUT_DIR / "plots").mkdir(parents=True, exist_ok=True)
(OUT_DIR / "reports").mkdir(parents=True, exist_ok=True)

## Input discovery and validation
Discover only PDFs under `car_pdfs/` and fail fast if the folder is missing or empty.

In [ ]:
def get_pdf_paths(pdf_dir: Path) -> List[Path]:
    return sorted([p for p in pdf_dir.glob("*.pdf") if p.is_file()])

PDF_DIR = Path(CONFIG["PDF_DIR"])
if not PDF_DIR.exists() or not PDF_DIR.is_dir():
    raise FileNotFoundError(f"PDF_DIR does not exist or is not a directory: {PDF_DIR}")

pdf_paths = get_pdf_paths(PDF_DIR)
if not pdf_paths:
    raise FileNotFoundError(f"No PDF files found in: {PDF_DIR}")

print(f"Found {len(pdf_paths)} car insurance PDFs in {PDF_DIR.resolve()}")

## Core helper functions
Text cleaning, header/footer detection, layout merging, and type heuristics.

In [ ]:
def light_clean_text(text: str) -> str:
    if not text:
        return ""
    t = text.replace("­", "")
    t = re.sub(r"-\s*\n\s*", "", t)
    t = t.replace("", "
")
    t = re.sub(r"[ 	]+", " ", t)
    t = re.sub(r"
{3,}", "

", t)
    return t.strip()


def split_lines_clean(text: str) -> List[str]:
    return [ln.strip() for ln in text.splitlines() if ln.strip()]


def detect_repeated_header_footer_lines(
    page_texts: List[str], min_page_frac: float = 0.5, max_chars: int = 80
) -> List[str]:
    n_pages = len(page_texts)
    if n_pages == 0:
        return []

    line_docfreq: Dict[str, int] = {}
    for text in page_texts:
        for ln in set(split_lines_clean(text)):
            if len(ln) <= max_chars:
                line_docfreq[ln] = line_docfreq.get(ln, 0) + 1

    threshold = max(2, math.ceil(min_page_frac * n_pages))
    return sorted([ln for ln, df in line_docfreq.items() if df >= threshold])


def remove_lines(text: str, remove_set: set) -> str:
    if not remove_set:
        return text
    lines = [ln for ln in split_lines_clean(text) if ln not in remove_set]
    return "
".join(lines).strip()


def blocks_reading_order(blocks: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    return sorted(blocks, key=lambda b: (round(float(b["y0"]), 1), round(float(b["x0"]), 1)))


def merge_blocks(blocks: List[Dict[str, Any]], vertical_gap: float, x_tol: float) -> List[List[Dict[str, Any]]]:
    groups: List[List[Dict[str, Any]]] = []
    current: List[Dict[str, Any]] = []

    for block in blocks:
        if not current:
            current = [block]
            continue

        prev = current[-1]
        close_vertically = (float(block["y0"]) - float(prev["y1"])) <= vertical_gap
        aligned = abs(float(block["x0"]) - float(prev["x0"])) <= x_tol

        if close_vertically and aligned:
            current.append(block)
        else:
            groups.append(current)
            current = [block]

    if current:
        groups.append(current)
    return groups


def classify_layout_type(text: str) -> str:
    first_line = split_lines_clean(text)[0] if split_lines_clean(text) else ""
    if re.match(r"^(hoofdstuk|artikel)", first_line, flags=re.IGNORECASE):
        return "heading"
    if re.match(r"^(\d+[\.)]|[-•])\s+", first_line):
        return "list_item"
    return "paragraph"


def merge_bbox(blocks: List[Dict[str, Any]]) -> List[float]:
    return [
        float(min(b["x0"] for b in blocks)),
        float(min(b["y0"] for b in blocks)),
        float(max(b["x1"] for b in blocks)),
        float(max(b["y1"] for b in blocks)),
    ]

## PDF extraction
Extract document/page/block records from each PDF.

In [ ]:
@dataclass
class BlockRecord:
    doc_id: str
    filename: str
    page_no: int
    block_id: str
    x0: float
    y0: float
    x1: float
    y1: float
    text_raw_block: str


def extract_pdf(doc_path: Path, config: Dict[str, Any]) -> Tuple[Dict[str, Any], List[Dict[str, Any]], List[BlockRecord], List[str]]:
    doc_id = doc_path.stem
    pages_records: List[Dict[str, Any]] = []
    block_records: List[BlockRecord] = []
    page_texts_for_repetition: List[str] = []

    try:
        pdf = fitz.open(doc_path)
        page_count = len(pdf)
        total_chars = 0

        for pidx in range(page_count):
            page = pdf[pidx]
            page_no = pidx + 1
            page_text_clean = light_clean_text(page.get_text("text") or "")
            page_texts_for_repetition.append(page_text_clean)

            page_char_count = len(page_text_clean)
            total_chars += page_char_count

            for bidx, b in enumerate(page.get_text("blocks")):
                x0, y0, x1, y1, btext, *_ = b
                btext_clean = light_clean_text(btext or "")
                if not btext_clean:
                    continue
                block_records.append(
                    BlockRecord(
                        doc_id=doc_id,
                        filename=doc_path.name,
                        page_no=page_no,
                        block_id=f"{doc_id}_p{page_no}_b{bidx}",
                        x0=float(x0), y0=float(y0), x1=float(x1), y1=float(y1),
                        text_raw_block=btext_clean,
                    )
                )

            pages_records.append(
                {
                    "doc_id": doc_id,
                    "filename": doc_path.name,
                    "page_no": page_no,
                    "page_char_count": page_char_count,
                    "page_text_raw": page_text_clean,
                }
            )

        low_text_pages = sum(r["page_char_count"] < config["PAGE_LOW_TEXT_CHARS"] for r in pages_records)
        doc_record = {
            "doc_id": doc_id,
            "filename": doc_path.name,
            "page_count": page_count,
            "char_count_total": total_chars,
            "extraction_ok": True,
            "low_text_page_frac": (low_text_pages / page_count) if page_count else 1.0,
            "suspect_scan": ((total_chars < config["DOC_MIN_TOTAL_CHARS"]) or ((low_text_pages / page_count) > config["MAX_LOW_TEXT_PAGE_FRAC"])) if page_count else True,
        }
        return doc_record, pages_records, block_records, page_texts_for_repetition

    except Exception as exc:
        return {
            "doc_id": doc_id,
            "filename": doc_path.name,
            "page_count": 0,
            "char_count_total": 0,
            "extraction_ok": False,
            "low_text_page_frac": 1.0,
            "suspect_scan": True,
            "error": repr(exc),
        }, [], [], []


document_rows: List[Dict[str, Any]] = []
pages_rows: List[Dict[str, Any]] = []
block_rows: List[Dict[str, Any]] = []
header_footer_map: Dict[str, Dict[str, Any]] = {}

for pdf_path in tqdm(pdf_paths, desc="Extracting PDFs"):
    doc_row, page_rows, block_recs, page_texts = extract_pdf(pdf_path, CONFIG)
    document_rows.append(doc_row)
    pages_rows.extend(page_rows)
    block_rows.extend([b.__dict__ for b in block_recs])

    repeated_lines = detect_repeated_header_footer_lines(
        page_texts,
        min_page_frac=CONFIG["REPEATED_LINE_PAGE_FRAC"],
        max_chars=CONFIG["REPEATED_LINE_MAX_CHARS"],
    )
    header_footer_map[doc_row["doc_id"]] = {
        "repeated_lines": repeated_lines,
        "n_repeated_lines": len(repeated_lines),
    }

documents_df = pd.DataFrame(document_rows)
pages_df = pd.DataFrame(pages_rows)
blocks_df = pd.DataFrame(block_rows)

print(documents_df[["doc_id", "extraction_ok", "page_count", "char_count_total"]].head())

## Header/footer detection and cleaning
Create cleaned page text by removing repeated short lines detected within each document.

In [ ]:
if not pages_df.empty:
    pages_df["suspected_header_footer_lines"] = pages_df["doc_id"].map(
        lambda d: header_footer_map.get(d, {}).get("repeated_lines", [])
    )
    pages_df["page_text_no_hf"] = pages_df.apply(
        lambda r: remove_lines(r["page_text_raw"], set(r["suspected_header_footer_lines"])),
        axis=1,
    )

## Atomic unit construction
Build atomic text units from merged layout blocks, maintaining section path and heading context.

In [ ]:
def build_atomic_units(
    doc_id: str,
    pages_df_doc: pd.DataFrame,
    blocks_df_doc: pd.DataFrame,
    repeated_lines: set,
    config: Dict[str, Any],
) -> List[Dict[str, Any]]:
    units: List[Dict[str, Any]] = []
    section_stack: List[str] = []
    pending_heading: Optional[Dict[str, Any]] = None

    for page_no in sorted(pages_df_doc["page_no"].unique().tolist()):
        page_blocks = blocks_df_doc[blocks_df_doc["page_no"] == page_no].copy()
        if page_blocks.empty:
            continue

        page_blocks["text_clean_no_hf"] = page_blocks["text_raw_block"].apply(
            lambda t: light_clean_text(remove_lines(t, repeated_lines))
        )
        page_blocks = page_blocks[page_blocks["text_clean_no_hf"].str.len() > 0]
        if page_blocks.empty:
            continue

        ordered_blocks = blocks_reading_order([
            {
                "block_id": row.block_id,
                "x0": row.x0,
                "y0": row.y0,
                "x1": row.x1,
                "y1": row.y1,
                "text": row.text_clean_no_hf,
            }
            for row in page_blocks.itertuples(index=False)
        ])

        merged_groups = merge_blocks(
            ordered_blocks,
            vertical_gap=config["BLOCK_MERGE_VERTICAL_GAP"],
            x_tol=config["BLOCK_MERGE_X_TOL"],
        )

        for group in merged_groups:
            text = light_clean_text("
".join(b["text"] for b in group))
            if not text:
                continue

            chunk_type_layout = classify_layout_type(text)
            first_line = split_lines_clean(text)[0] if split_lines_clean(text) else ""
            if chunk_type_layout == "heading" and first_line:
                section_stack = section_stack[:3]
                section_stack.append(first_line)

            unit = {
                "doc_id": doc_id,
                "page_no": page_no,
                "text": text,
                "block_ids": [b["block_id"] for b in group],
                "bbox": merge_bbox(group),
                "section_path": " > ".join(section_stack) if section_stack else "",
                "chunk_type_layout": chunk_type_layout,
            }

            if chunk_type_layout == "heading":
                pending_heading = unit
                continue

            if pending_heading is not None:
                unit["text"] = f"{pending_heading['text']}
{unit['text']}"
                unit["block_ids"] = pending_heading["block_ids"] + unit["block_ids"]
                unit["bbox"] = [
                    min(pending_heading["bbox"][0], unit["bbox"][0]),
                    min(pending_heading["bbox"][1], unit["bbox"][1]),
                    max(pending_heading["bbox"][2], unit["bbox"][2]),
                    max(pending_heading["bbox"][3], unit["bbox"][3]),
                ]
                if not unit["section_path"] and pending_heading["section_path"]:
                    unit["section_path"] = pending_heading["section_path"]
                pending_heading = None

            units.append(unit)

    if pending_heading is not None:
        units.append(pending_heading)

    return units

## Overlapping chunk construction
Build sliding-window chunks over atomic units with configurable target/min/overlap settings.

In [ ]:
def make_overlapping_chunks(doc_id: str, units: List[Dict[str, Any]], config: Dict[str, Any]) -> List[Dict[str, Any]]:
    chunks: List[Dict[str, Any]] = []
    if not units:
        return chunks

    target_chars = config["CHUNK_TARGET_CHARS"]
    min_chars = config["CHUNK_MIN_CHARS"]
    overlap_chars = config["CHUNK_OVERLAP_CHARS"]
    allow_cross_page = config["ALLOW_CROSS_PAGE_CHUNKS"]

    start = 0
    order_index = 0
    n = len(units)

    while start < n:
        current_units: List[Dict[str, Any]] = []
        current_chars = 0
        start_page = units[start]["page_no"]

        i = start
        while i < n:
            u = units[i]
            if not allow_cross_page and current_units and u["page_no"] != start_page:
                break

            u_len = len(u["text"])
            if current_units and current_chars >= target_chars:
                break

            current_units.append(u)
            current_chars += u_len + 2
            i += 1

        if not current_units:
            start += 1
            continue

        if current_chars < min_chars and i < n:
            while i < n:
                u = units[i]
                if not allow_cross_page and u["page_no"] != start_page:
                    break
                current_units.append(u)
                current_chars += len(u["text"]) + 2
                i += 1
                if current_chars >= min_chars:
                    break

        text_raw = "

".join(u["text"] for u in current_units).strip()
        block_ids = [bid for u in current_units for bid in u["block_ids"]]
        x0 = min(u["bbox"][0] for u in current_units)
        y0 = min(u["bbox"][1] for u in current_units)
        x1 = max(u["bbox"][2] for u in current_units)
        y1 = max(u["bbox"][3] for u in current_units)
        section_path = next((u["section_path"] for u in current_units if u["section_path"]), "")

        chunks.append(
            {
                "doc_id": doc_id,
                "chunk_id": f"{doc_id}_c{order_index:05d}",
                "page_start": int(current_units[0]["page_no"]),
                "page_end": int(current_units[-1]["page_no"]),
                "block_ids": block_ids,
                "bbox": [x0, y0, x1, y1],
                "section_path": section_path,
                "chunk_type_layout": current_units[0]["chunk_type_layout"],
                "order_index": int(order_index),
                "text_raw": text_raw,
            }
        )

        order_index += 1

        if i >= n:
            break

        back_chars = 0
        next_start = len(current_units) - 1
        while next_start > 0 and back_chars < overlap_chars:
            back_chars += len(current_units[next_start]["text"]) + 2
            next_start -= 1
        start += max(1, next_start + 1)

    return chunks


def chunk_document(doc_id: str, pages_df_doc: pd.DataFrame, blocks_df_doc: pd.DataFrame, config: Dict[str, Any]) -> List[Dict[str, Any]]:
    repeated_lines = set(header_footer_map.get(doc_id, {}).get("repeated_lines", []))
    units = build_atomic_units(doc_id, pages_df_doc, blocks_df_doc, repeated_lines, config)
    return make_overlapping_chunks(doc_id, units, config)


chunk_rows: List[Dict[str, Any]] = []
for doc_id in tqdm(documents_df.loc[documents_df["extraction_ok"], "doc_id"].tolist(), desc="Chunking docs"):
    chunk_rows.extend(
        chunk_document(
            doc_id,
            pages_df_doc=pages_df[pages_df["doc_id"] == doc_id],
            blocks_df_doc=blocks_df[blocks_df["doc_id"] == doc_id],
            config=CONFIG,
        )
    )

chunks_df = pd.DataFrame(chunk_rows)
print(f"Built {len(chunks_df)} chunks")

## Optional enrichments
Add optional lexical text and a lightweight section summary for diagnostics.

In [ ]:
NLP = None
if SPACY_AVAILABLE:
    try:
        NLP = spacy.load("nl_core_news_sm", disable=["parser", "ner", "textcat"])
    except Exception as exc:
        warnings.warn(f"spaCy model unavailable; text_lexical disabled: {exc}")


def build_text_lexical(text: str, max_chars: int = 2000) -> str:
    if NLP is None or not isinstance(text, str) or not text.strip():
        return ""
    doc = NLP(text[:max_chars])
    return " ".join(
        token.lemma_.strip().lower()
        for token in doc
        if not (token.is_space or token.is_punct or token.is_stop)
    )

if not chunks_df.empty:
    chunks_df["text_lexical"] = [
        build_text_lexical(t, max_chars=CONFIG["MAX_CHARS_FOR_SPACY"])
        for t in tqdm(chunks_df["text_raw"].fillna(""), desc="Lexical preprocessing")
    ]

    section_summary_df = (
        chunks_df.assign(section_path=chunks_df["section_path"].fillna(""))
        .groupby(["doc_id", "section_path"], dropna=False)
        .agg(n_chunks=("chunk_id", "count"), chars=("text_raw", lambda s: int(s.str.len().sum())))
        .reset_index()
        .sort_values(["doc_id", "n_chunks"], ascending=[True, False])
    )
else:
    section_summary_df = pd.DataFrame(columns=["doc_id", "section_path", "n_chunks", "chars"])

## Output saving
Persist canonical outputs: `documents.csv`, `pages.csv`, `chunks.parquet`.

In [ ]:
def json_serialize_if_needed(value: Any) -> Any:
    if isinstance(value, (list, dict)):
        return json.dumps(value, ensure_ascii=False)
    return value

pages_to_save = pages_df.copy()
if not CONFIG["SAVE_FULL_PAGE_TEXT"] and "page_text_raw" in pages_to_save.columns:
    pages_to_save = pages_to_save.drop(columns=["page_text_raw"])

for col in ["suspected_header_footer_lines"]:
    if col in pages_to_save.columns:
        pages_to_save[col] = pages_to_save[col].map(json_serialize_if_needed)

for col in ["block_ids", "bbox"]:
    if col in chunks_df.columns:
        chunks_df[col] = chunks_df[col].map(json_serialize_if_needed)

documents_path = OUT_DIR / "documents.csv"
pages_path = OUT_DIR / "pages.csv"
chunks_path = OUT_DIR / "chunks.parquet"

if not documents_df.empty:
    documents_df.to_csv(documents_path, index=False)
if not pages_to_save.empty:
    pages_to_save.to_csv(pages_path, index=False)
if not chunks_df.empty:
    chunks_df.to_parquet(chunks_path, index=False)

with open(OUT_DIR / "header_footer.json", "w", encoding="utf-8") as f:
    json.dump(header_footer_map, f, ensure_ascii=False, indent=2)

if not section_summary_df.empty:
    section_summary_df.to_csv(OUT_DIR / "reports" / "cluster_summary.csv", index=False)

print(f"Saved outputs to {OUT_DIR.resolve()}")

## QA / diagnostics
Basic sanity checks and a chunk length histogram.

In [ ]:
qa_summary = {
    "n_documents": int(len(documents_df)),
    "n_pages": int(len(pages_df)),
    "n_chunks": int(len(chunks_df)),
    "failed_docs": int((~documents_df["extraction_ok"]).sum()) if not documents_df.empty else 0,
    "median_chunk_chars": float(chunks_df["text_raw"].str.len().median()) if not chunks_df.empty else 0.0,
}

with open(OUT_DIR / "reports" / "qa_summary.json", "w", encoding="utf-8") as f:
    json.dump(qa_summary, f, indent=2)

if not chunks_df.empty:
    plt.figure(figsize=(8, 5))
    plt.hist(chunks_df["text_raw"].str.len(), bins=30)
    plt.title("Chunk length distribution")
    plt.xlabel("Characters")
    plt.ylabel("Frequency")
    plt.tight_layout()
    plt.savefig(OUT_DIR / "plots" / "chunk_char_hist.png", dpi=140)
    plt.close()

qa_summary